In [79]:
import numpy as np
import pandas as pd
import torch
from torch import nn 

In [80]:
def train_test_split(X, y, test_size:float = None, train_size:float = None):
	if X.shape[0] != y.shape[0]:
		raise Exception("Size mismatch of X and y")
	else:
		total_rows=X.shape[0]
	if test_size == None and train_size == None:
		test_size=0.25
		train_size = 1 - test_size
	elif test_size == None:
		test_size = 1 - train_size
	elif train_size == None:
		train_size = 1 - test_size

	mark = int(total_rows*train_size)
	return X[:mark], y[:mark], X[mark:], y[mark:]


In [81]:
# from sklearn.model_selection import train_test_split
df = pd.read_csv('StudentPerformance.csv')
X = df.drop(columns=['Performance Index', 'Extracurricular Activities']).to_numpy()
y = df['Performance Index'].to_numpy()

# c1 = df['Hours Studied'].to_numpy().reshape(-1, 1)
# c2 = df['Previous Scores'].to_numpy().reshape(-1, 1)
# c3 = df['Sleep Hours'].to_numpy().reshape(-1, 1)
# c4 = df['Sample Question Papers Practiced'].to_numpy().reshape(-1, 1)

df.head()


,Hours Studied,Previous Scores,Extracurricular Activities,Sleep Hours,Sample Question Papers Practiced,Performance Index
0,7,99,Yes,9,1,91.0
1,4,82,No,4,2,65.0
2,8,51,Yes,7,2,45.0
3,5,52,Yes,5,2,36.0
4,7,75,No,8,5,66.0


In [82]:

X_train, y_train, X_test, y_test = train_test_split(X, y, 0.2)
# Scaling
from sklearn.preprocessing import StandardScaler
# scaler1 = StandardScaler()
# scaler2 = StandardScaler()
# scaler3 = StandardScaler()
# scaler4 = StandardScaler()

# c1 = scaler1.fit_transform(c1)
# c2 = scaler2.fit_transform(c2)
# c3 = scaler3.fit_transform(c3)
# c4 = scaler4.fit_transform(c4)
# X_train = scaler.fit_transform(X_train)
# X_test = scaler.transform(X_test)

X_train

array([[ 7, 99,  9,  1],
       [ 4, 82,  4,  2],
       [ 8, 51,  7,  2],
       ...,
       [ 5, 86,  8,  8],
       [ 7, 53,  6,  3],
       [ 3, 82,  8,  3]], shape=(8000, 4))

In [ ]:
class Perceptron():
	def __init__(self, features_size):
		self.weights = torch.tensor(np.random.random(features_size), requires_grad=True, dtype=torch.float32)
		self.bias = torch.zeros(1, requires_grad=True, dtype=torch.float32)
		self.features_size = features_size
		self.m_weights = torch.zeros_like(self.weights)
		self.v_weights = torch.zeros_like(self.weights)
		self.m_bias = torch.zeros_like(self.bias)
		self.v_bias = torch.zeros_like(self.bias)
		self.t = 0

	def forward(self, features):
		features = torch.tensor(features, dtype=torch.float32)
		# self.model_pred = torch.empty(features.shape[0], dtype=torch.float32)
		self.model_pred = (features @ self.weights) + self.bias
		return self.model_pred
			
	def rmse(self, y_true):
		y_true = torch.tensor(y_true, dtype=torch.float32)
		if y_true.shape != self.model_pred.shape:
			raise Exception("Shape mismatch(y_true and y_pred are not same shape)")
		# num = y_true.numel()
		self.loss = torch.mean((y_true-self.model_pred)**2).sqrt()
		return self.loss
	# Vanilla Gradient Descent
	def backpropagation(self, lr=0.01):
		with torch.no_grad():
			# for i in range(len(self.weights)):
			# 	self.weights[i] = self.weights[i] - (lr * self.weights.grad[i])
			self.weights -= lr * self.weights.grad
			self.bias -= (lr * self.bias.grad)
	# Adam Optimizer for backpropagation/weight updates
	def adam(self, beta1, beta2, eps=1e-8, lr=0.001):
		self.t += 1
		with torch.no_grad():
			for i in range(self.weights.shape[0]):
				self.m_weights[i] = beta1 * self.m_weights[i] + (1 - beta1) * self.weights.grad[i]
				self.v_weights[i] = beta2 * self.v_weights[i] + (1 - beta2) * self.weights.grad[i]**2


				mhat_w = self.m_weights[i] / (1 - beta1**self.t)
				vhat_w = self.v_weights[i] / (1 - beta2**self.t)

				self.weights[i] -= lr*mhat_w/(vhat_w**0.5 + eps)
			for i in range(self.bias.shape[0]):
				self.m_bias[i] = beta1 * self.m_bias[i] + (1 - beta1) * self.bias.grad[0]
				self.v_bias[i] = beta2 * self.v_bias[i] + (1 - beta2) * self.bias.grad[0]**2

				mhat_b = self.m_bias[i] / (1 - beta1**self.t)
				vhat_b = self.v_bias[i] / (1 - beta2**self.t)

				self.bias[i] -= lr*mhat_b/(vhat_b**0.5 + eps)
				
	def zero_grad(self):
		self.weights.grad = None
		self.bias.grad = None

In [84]:
from torch.utils.data import Dataset, DataLoader
class CustomDataset(Dataset):
	def __init__(self, features, labels):
		self.features = features
		self.labels = labels
	
	def __len__(self):
		return len(self.features)
	
	def __getitem__(self, idx):
		return self.features[idx], self.labels[idx]

In [85]:
train_set = CustomDataset(X_train, y_train)

In [86]:
train_loader = DataLoader(train_set, batch_size=32, shuffle=True)

In [87]:
model = Perceptron(X_train.shape[1])
epochs = 20
y_pred_model = []
for i in range(epochs):
	losses=[]
	for batch_feature, batch_label in train_loader:

		model.zero_grad()

		y_pred = (model.forward(batch_feature))

		# Calculate loss

		loss = model.rmse(batch_label)
		losses.append(loss)
		# print("loss:", loss)
		
		# Calculating loss
		model.loss.backward()

		# update weights 
		# model.backpropagation(lr=0.01)
		model.adam(beta1=0.9, beta2=0.999, lr=0.001)
		
	print(f"Avg loss in epoch {i + 1}: {(sum(losses)/len(losses)):.3f}")
	losses.clear()


C:\Users\LOQ\AppData\Local\Temp\ipykernel_17240\845225702.py:13: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  features = torch.tensor(features, dtype=torch.float32)
C:\Users\LOQ\AppData\Local\Temp\ipykernel_17240\845225702.py:19: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y_true = torch.tensor(y_true, dtype=torch.float32)


Avg loss in epoch 1: 36.520
Avg loss in epoch 2: 17.195
Avg loss in epoch 3: 9.230
Avg loss in epoch 4: 8.737
Avg loss in epoch 5: 8.476
Avg loss in epoch 6: 8.206
Avg loss in epoch 7: 7.921
Avg loss in epoch 8: 7.648
Avg loss in epoch 9: 7.381
Avg loss in epoch 10: 7.138
Avg loss in epoch 11: 6.904
Avg loss in epoch 12: 6.693
Avg loss in epoch 13: 6.502
Avg loss in epoch 14: 6.333
Avg loss in epoch 15: 6.192
Avg loss in epoch 16: 6.067
Avg loss in epoch 17: 5.958
Avg loss in epoch 18: 5.855
Avg loss in epoch 19: 5.773
Avg loss in epoch 20: 5.700


In [88]:
import numpy as np
y_model=[]
with torch.no_grad():
	for x in X_test:
		y = model.forward(x)
		y_model.append(y)

y_model = np.array(y_model)
y_model

array([[38.17692 ],
       [39.146763],
       [58.587017],
       ...,
       [67.746826],
       [87.528534],
       [63.93043 ]], shape=(2000, 1), dtype=float32)

In [89]:
y_test

array([28., 35., 63., ..., 74., 95., 64.], shape=(2000,))

In [90]:
from sklearn.metrics import r2_score
score = r2_score(y_test, y_model)
score

0.9124490981666782